<a href="https://colab.research.google.com/github/gustavopatinot/proof_wandb/blob/main/notebooks/exp_datascience_wandb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Como lograr la interacción entre Colab y W&B

# Dashboard

In [1]:
!pip install wandb -q #install library
import wandb

In [4]:
from google.colab import userdata
import wandb

# Retrieve the secret key
wandb_key = userdata.get('WANDB_API_KEY')

# Login to Weights & Biases
wandb.login(key=wandb_key)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gustavopatinot (gustavopatinot-universidad-santo-tomas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
project_name = 'first_steps'
group_name = 'cnn'
experiment_name = '2_conv'

In [7]:
import tensorflow as tf
from tensorflow.keras.callbacks import Callback
# Updated import for newer wandb versions
from wandb.integration.keras import WandbCallback
import numpy as np

# Primer parámetro configurado

- Importante tener presente la configuración de cada experimento.

In [8]:
wandb.init(
    project=project_name,
    group=group_name,
    name=experiment_name,
    config={
        "conv_1": 16,
        "activation_1": "relu",
        "kernel_size": (3, 3),
        "pool_size": (2, 2),
        "dropout": 0.7,
        "conv_2": 32,
        "activation_out": "softmax",
        "optimizer": "adam",
        "loss": "sparse_categorical_crossentropy",
        "metric": "accuracy",
        "epoch": 6,
        "batch_size": 32
    })
config = wandb.config

In [9]:
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data() ##data download
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
x_train, y_train = x_train[::5], y_train[::5]

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [10]:
class_names = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [11]:
def cnn_mnist(config, num_classes = 10, input_shape = (28, 28, 1)): ##simple Keras CNN
    img_inputs = tf.keras.Input(shape=input_shape)
    conv_1 = tf.keras.layers.Conv2D(config.conv_1, kernel_size=config.kernel_size, activation=config.activation_1)(img_inputs)
    pool_1 = tf.keras.layers.MaxPooling2D(pool_size=config.pool_size)(conv_1)
    conv_2 = tf.keras.layers.Conv2D(config.conv_2, kernel_size=config.kernel_size, activation=config.activation_1)(pool_1)
    pool_2 = tf.keras.layers.MaxPooling2D(pool_size=config.pool_size)(conv_2)
    flatten = tf.keras.layers.Flatten()(pool_2)
    dropout = tf.keras.layers.Dropout(config.dropout)(flatten)
    dense_out = tf.keras.layers.Dense(num_classes, activation=config.activation_out)(dropout)
    model = tf.keras.models.Model(inputs=img_inputs, outputs=dense_out)
    model.compile(loss=config.loss, optimizer=config.optimizer, metrics=[config.metric])
    return model

In [12]:
our_model = cnn_mnist(config=config)

In [16]:
from wandb.integration.keras import WandbCallback

our_model.fit(
    x_train,
    y_train,
    epochs=config.epoch,
    batch_size=config.batch_size,
    validation_data=(x_test, y_test),
    callbacks=[
        WandbCallback(save_model=False, save_graph=False)
    ]
)

wandb.finish()

Epoch 1/6
375/375 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7218 - loss: 0.8385 - val_accuracy: 0.9438 - val_loss: 0.2071
Epoch 2/6
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9059 - loss: 0.3102 - val_accuracy: 0.9576 - val_loss: 0.1463
Epoch 3/6
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9288 - loss: 0.2348 - val_accuracy: 0.9651 - val_loss: 0.1192
Epoch 4/6
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9352 - loss: 0.2154 - val_accuracy: 0.9697 - val_loss: 0.1057
Epoch 5/6
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9468 - loss: 0.1812 - val_accuracy: 0.9730 - val_loss: 0.0937
Epoch 6/6
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9500 - loss: 0.1694 - val_accuracy: 0.9759 - val_loss: 0.0817


accuracy,▁▇▇███
epoch,▁▂▄▅▇█
loss,█▂▂▁▁▁
val_accuracy,▁▄▆▇▇█
val_loss,█▅▃▂▂▁
accuracy,0.95
best_epoch,5
best_val_loss,0.08168
epoch,5
loss,0.1694
val_accuracy,0.9759


# Segunda Configuración

In [17]:
project_name = 'first_steps'
group_name = 'cnn'
experiment_name = '2_conv_changed_channels'

wandb.init(
    project=project_name,
    group=group_name,
    name=experiment_name,
    config={
        "conv_1": 16,
        "activation_1": "relu",
        "kernel_size": (3, 3),
        "pool_size": (2, 2),
        "dropout": 0.7,
        "conv_2": 32,
        "activation_out": "softmax",
        "optimizer": "adam",
        "loss": "sparse_categorical_crossentropy",
        "metric": "accuracy",
        "epoch": 6,
        "batch_size": 32
    })
config = wandb.config

our_model = cnn_mnist(config=config)

In [19]:
from wandb.integration.keras import WandbCallback

our_model.fit(x_train, y_train, epochs=config.epoch, batch_size=config.batch_size,
          validation_data=(x_test, y_test),
          callbacks=[WandbCallback(data_type="image",
                                   labels=class_names,
                                   save_model=False,
                                   save_graph=False)])

wandb.finish()

Epoch 1/6
374/375 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5182 - loss: 1.4452

wandb: WARNING No validation_data set, pass a generator to the callback.


375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7147 - loss: 0.8847 - val_accuracy: 0.9387 - val_loss: 0.2284
Epoch 2/6
369/375 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8936 - loss: 0.3337

wandb: WARNING No validation_data set, pass a generator to the callback.


375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9033 - loss: 0.3059 - val_accuracy: 0.9591 - val_loss: 0.1506
Epoch 3/6
367/375 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9275 - loss: 0.2391

...

wandb: WARNING No validation_data set, pass a generator to the callback.


375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9427 - loss: 0.1849 - val_accuracy: 0.9736 - val_loss: 0.0907
Epoch 6/6
374/375 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9491 - loss: 0.1639

wandb: WARNING No validation_data set, pass a generator to the callback.


375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9492 - loss: 0.1617 - val_accuracy: 0.9747 - val_loss: 0.0828


accuracy,▁▇▇███
epoch,▁▂▄▅▇█
loss,█▂▂▁▁▁
val_accuracy,▁▅▆▇██
val_loss,█▄▃▂▁▁
accuracy,0.94925
best_epoch,5
best_val_loss,0.0828
epoch,5
loss,0.1617
val_accuracy,0.9747


# Barridos - Sweeps

In [21]:
# Configurar el barrido: especificar los parámetros a buscar, la estrategia de búsqueda, la métrica de optimización, etc..
sweep_config = {
    'method': 'random', #grid, random
    'metric': {
      'name': 'accuracy',
      'goal': 'maximize'
    },
    'parameters': {
        'epoch': {
            'values': [5, 10]
        },
        'dropout': {
            'values': [0.3, 0.4, 0.5]
        },
        'conv_1': {
            'values': [16, 32, 64]
        },
        'conv_2': {
            'values': [16, 32, 64]
        },
        'optimizer': {
            'values': ['adam', 'nadam', 'sgd', 'rmsprop']
        },
        'activation_1': {
            'values': ['relu', 'elu', 'selu','sigmoid']
        },
        'kernel_size': {
            'values': [(3, 3), (5, 5), (7, 7)]
        },

    }
}

In [24]:
wandb.login(key=wandb_key)
# Fetch the actual entity name (username or team) from the API
api = wandb.Api()
# Corrected: removed () from viewer
user_name = api.viewer.entity

sweep_id = wandb.sweep(sweep_config, entity=user_name, project="first_steps")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Create sweep with ID: bcjopon5
Sweep URL: https://wandb.ai/gustavopatinot-universidad-santo-tomas/first_steps/sweeps/bcjopon5


In [27]:
def train():
    # Valores predeterminados para los hiperparámetros
    config_defaults = {
        "conv_1": 32,
        "activation_1": "relu",
        "kernel_size": (3, 3),
        "pool_size": (2, 2),
        "dropout": 0.1,
        "conv_2": 64,
        "activation_out": "softmax",
        "optimizer": "adam",
        "loss": "sparse_categorical_crossentropy",
        "metric": "accuracy",
        "epoch": 6,
        "batch_size": 32
    }

    # Inicializar un nuevo wandb run
    wandb.init(config=config_defaults)
    config = wandb.config

    model = cnn_mnist(config=config)

    # Use the integration callback with Keras 3 compatible settings
    from wandb.integration.keras import WandbCallback
    model.fit(x_train, y_train, epochs=config.epoch, batch_size=config.batch_size,
          validation_data=(x_test, y_test),
          callbacks=[WandbCallback(save_model=False, save_graph=False)])

In [28]:
wandb.agent(sweep_id, train)

wandb: Agent Starting Run: 7mpedpu8 with config:
wandb: 	activation_1: elu
wandb: 	conv_1: 64
wandb: 	conv_2: 64
wandb: 	dropout: 0.3
wandb: 	epoch: 10
wandb: 	kernel_size: [3, 3]
wandb: 	optimizer: rmsprop
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/10


375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8811 - loss: 0.3853 - val_accuracy: 0.9582 - val_loss: 0.1451
Epoch 2/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9589 - loss: 0.1430 - val_accuracy: 0.9698 - val_loss: 0.0995
Epoch 3/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9703 - loss: 0.0998 - val_accuracy: 0.9728 - val_loss: 0.0841
Epoch 4/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9758 - loss: 0.0787 - val_accuracy: 0.9823 - val_loss: 0.0609
Epoch 5/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9797 - loss: 0.0697 - val_accuracy: 0.9786 - val_loss: 0.0684
Epoch 6/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9827 - loss: 0.0593 - val_accuracy: 0.9815 - val_loss: 0.0627
Epoch 7/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9828 - loss: 0.0500 - val_accuracy: 0.9786 - val_loss: 0.0784
Epoch 8/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9858 - loss: 0.0450 - val_accuracy: 0.9809 - val_

accuracy,▁▆▇▇▇█████
epoch,▁▂▃▃▄▅▆▆▇█
loss,█▃▂▂▂▁▁▁▁▁
val_accuracy,▁▄▅█▇█▇███
val_loss,█▄▃▁▂▁▂▁▂▁
accuracy,0.98842
best_epoch,3
best_val_loss,0.06093
epoch,9
loss,0.03671
val_accuracy,0.9825


wandb: Agent Starting Run: gyauz4os with config:
wandb: 	activation_1: selu
wandb: 	conv_1: 32
wandb: 	conv_2: 32
wandb: 	dropout: 0.5
wandb: 	epoch: 10
wandb: 	kernel_size: [5, 5]
wandb: 	optimizer: nadam
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


...

accuracy,▁▆▇▇▇▇████
epoch,▁▂▃▃▄▅▆▆▇█
loss,█▃▂▂▂▁▁▁▁▁
val_accuracy,▁▄▆▆▆▇▇▇██
val_loss,█▅▃▃▂▂▂▂▁▁
accuracy,0.97367
best_epoch,9
best_val_loss,0.06648
epoch,9
loss,0.08887
val_accuracy,0.9798


wandb: Agent Starting Run: 8q6oh62f with config:
wandb: 	activation_1: sigmoid
wandb: 	conv_1: 16
wandb: 	conv_2: 16
wandb: 	dropout: 0.5
wandb: 	epoch: 5
wandb: 	kernel_size: [7, 7]
wandb: 	optimizer: rmsprop
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.2689 - loss: 2.0438 - val_accuracy: 0.7327 - val_loss: 1.2534
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6661 - loss: 1.0772 - val_accuracy: 0.8521 - val_loss: 0.6435
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7731 - loss: 0.7315 - val_accuracy: 0.8929 - val_loss: 0.4479
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8193 - loss: 0.5879 - val_accuracy: 0.9129 - val_loss: 0.3506
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8429 - loss: 0.5087 - val_accuracy: 0.9287 - val_loss: 0.2946


accuracy,▁▆▇██
epoch,▁▃▅▆█
loss,█▄▂▁▁
val_accuracy,▁▅▇▇█
val_loss,█▄▂▁▁
accuracy,0.84292
best_epoch,4
best_val_loss,0.29459
epoch,4
loss,0.50872
val_accuracy,0.9287


wandb: Ctrl + C detected. Stopping sweep.


# Artifacts

In [29]:
from collections import namedtuple
Dataset = namedtuple("Dataset", ["x", "y"])

def load_data_split(train_size=50_000):


    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()


    x_train, x_val = x_train[:train_size], x_train[train_size:]
    y_train, y_val = y_train[:train_size], y_train[train_size:]

    training_data = Dataset(x_train, y_train)
    validation_data = Dataset(x_val, y_val)
    test_data = Dataset(x_test, y_test)

    datasets = [training_data, validation_data, test_data]

    return datasets

In [30]:
def load_and_log():
    with wandb.init(project=project_name, job_type="load-data") as run:

        datasets = load_data_split()
        names = ["training", "validation", "test"]

        # Artifact
        raw_data = wandb.Artifact(
            "mnist-raw", type="dataset",
            description="Raw MNIST dataset, splitted",
            metadata={"source": "keras.datasets.mnist",
                      "train_data": len(datasets[0].x),
                      "valid_data": len(datasets[1].x),
                      "test_daata": len(datasets[2].x)})

        for name, data in zip(names, datasets):
            # Save our datasets
            with raw_data.new_file(name + ".npz", mode="wb") as file:
                np.savez(file, x=data.x, y=data.y)
        #save Artifact
        run.log_artifact(raw_data)

load_and_log()

In [31]:
def preprocess_dataset(dataset, normalize=True, expand_dims=True, to_categorical=True):
    x, y = dataset.x, dataset.y

    if normalize:
        x = x.astype("float32") / 255

    if expand_dims:
        x = np.expand_dims(x, -1)

    if to_categorical:
        y = tf.keras.utils.to_categorical(y, num_classes)

    return Dataset(x, y)

In [33]:
import os
# Define missing variable needed by preprocess_dataset
num_classes = 10

def preprocess_and_log(preprocess_steps):

    with wandb.init(project=project_name, job_type="data_preprocessing", name="preprocess_simple") as run:

        processed_data = wandb.Artifact(
            "mnist-preprocessed", type="dataset",
            description="Preprocessed MNIST dataset",
            metadata=preprocess_steps)

        # which Artifact we will use
        raw_data_artifact = run.use_artifact('mnist-raw:latest')

        # download Artifact
        raw_dataset = raw_data_artifact.download()

        for split in ["training", "validation", "test"]:
            datafile = split + ".npz"
            data = np.load(os.path.join(raw_dataset, datafile))
            raw_split = Dataset(x=data["x"], y=data["y"])
            processed_dataset = preprocess_dataset(raw_split, **preprocess_steps)

            with processed_data.new_file(split + ".npz", mode="wb") as file:
                np.savez(file, x=processed_dataset.x, y=processed_dataset.y)

        run.log_artifact(processed_data)


steps = {"normalize": True,
         "expand_dims": True,
         "to_categorical" : True}

preprocess_and_log(steps)

wandb: Downloading large artifact 'mnist-raw:latest', 52.41MB. 3 files...
wandb:   3 of 3 files downloaded.  
Done. 00:00:00.4 (131.2MB/s)
